<a href="https://colab.research.google.com/github/InduM/2Dto3D/blob/main/text_to_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3D Object Generation from Images/Prompts

This notebook is inspired by the [PyImageSearch blog post](https://pyimagesearch.com/2024/11/25/create-a-3d-object-from-your-images-with-triposr-in-python/) by Ritwik Rava. It showcases how to create a 3D object using **TripoSR**, a powerful tool for reconstructing 3D geometry from 2D images.

### What You Can Do

You can generate a 3D model using either of the following methods:

1. **From Your Own Images:** Upload a 2D image which TripoSR will use to create a 3D representation.

2. **From Text Description (Prompt):**  
   - Provide a natural language description of the object.
   - The prompt is sent to **Stable Diffusion** by StabilityAI to generate a realistic 2D image.
   - The generated image is then passed to **TripoSR**, which reconstructs a 3D object from it.

### Tools Used
- **TripoSR**: For 3D surface reconstruction from single/multiple views.
- **Stable Diffusion**: For generating synthetic 2D images from text prompts.

Let's get started by setting up the environment.

Setting up ***environment*** (takes about 12 minutes). Make sure to uncomment before executing the cell

In [3]:
!git clone https://github.com/pyimagesearch/TripoSR.git
import sys
sys.path.append('/content/TripoSR/tsr')
%cd TripoSR
#!pip install -r requirements.txt -q
#!pip install onnxruntime open3d
###!pip install --upgrade pillow==9.0.0
#!pip install --upgrade Pillow


Cloning into 'TripoSR'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 164 (delta 65), reused 42 (delta 42), pack-reused 67 (from 1)
Receiving objects: 100% (164/164), 36.71 MiB | 41.72 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/TripoSR/TripoSR


Import dependencies

In [4]:
import torch
import os
import time
from PIL import Image
import numpy as np
from google.colab import files
from diffusers import StableDiffusionXLPipeline
from tsr.system import TSR
import pymeshlab as pymesh
import open3d as o3d
import rembg

Determine wheteher to use GPU/CPU


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
def text_to_image(prompt):
# Load the fast SDXL Turbo model
  pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
  ).to(device)

# Set inference parameters

  image = pipe(prompt=prompt, guidance_scale=0.0, num_inference_steps=1).images[0]
  image = image.convert("RGBA")

# Save or display
  image.save("input_image.png")
  return image

Choosing an input method

In [8]:
print("Choose an option:")
print("1.Upload an image")
print("2.Describe the image")
print("3.Exit")

choice = input("Enter your choice (1/2/3): ")

if choice == '1':
    uploaded = files.upload()
    original_image = Image.open(list(uploaded.keys())[0])
elif choice == '2':
    prompt = (input("Enter prompt for the image:"))
    torch.cuda.empty_cache()
    print("Generating image...")
    original_image = text_to_image(prompt)
elif choice == '3':
    print("Goodbye!")
else:
    print("Invalid choice. Please try again.")

Choose an option:
1.Upload an image
2.Describe the image
3.Exit
Enter your choice (1/2/3): 1


Saving flamingo.png to flamingo.png


Resize the Image

In [9]:
original_image.resize((512, 512)).save("examples/product.png")

Setting up the parameters for TripoSR

In [10]:
image_paths = "/content/TripoSR/examples/product.png"
device = "cuda:0"
pretrained_model_name_or_path = "stabilityai/TripoSR"
output_dir = "output/"
model_save_format = "obj"
output_dir = output_dir.strip()
os.makedirs(output_dir, exist_ok=True)

Initialize the TripoSR model


In [11]:
model = TSR.from_pretrained(
    pretrained_model_name_or_path,
    config_name="config.yaml",
    weight_name="model.ckpt",
)
model.to(device)
print("Finished initializing")

Finished initializing


Process the input image

In [12]:
## Function to remove the background in images
def remove_background2(input_image):
    input_array = np.array(input_image)
    output_array = rembg.remove(input_array)
    output_image = Image.fromarray(output_array)
    output_image = output_image.convert("RGBA")
    return output_image


In [13]:
images = []
image = remove_background2(original_image)
if image.mode == "RGBA":
    image = np.array(image).astype(np.float32) / 255.0
    image = image[:, :, :3] * image[:, :, 3:4] + (1 - image[:, :, 3:4]) * 0.5
    image = Image.fromarray((image * 255.0).astype(np.uint8))
image_dir = os.path.join(output_dir, str(0))
os.makedirs(image_dir, exist_ok=True)
image.save(os.path.join(image_dir, "input.png"))
images.append(image)

Generate 3D model

In [13]:
## to free up memory
#from numba import cuda
#gpu = cuda.get_current_device()
#gpu.reset()



In [15]:
for i, image in enumerate(images):
    print(f"Running image {i + 1}/{len(images)} ...")
    with torch.no_grad():
        scene_codes = model([image], device=device)

    print("Exporting mesh")
    meshes = model.extract_mesh(scene_codes, has_vertex_color=False)
    mesh_file = os.path.join(output_dir, str(i), f"mesh.{model_save_format}")
    meshes[0].export(mesh_file)
print("Processing complete.")

Running image 1/1 ...
Exporting mesh


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 14.74 GiB of which 920.12 MiB is free. Process 49537 has 13.84 GiB memory in use. Of the allocated memory 13.64 GiB is allocated by PyTorch, and 81.46 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

Visualization

In [16]:
def visualize_obj_model(model_path):
    """Visualizes a .obj 3D model using Open3D."""
    mesh = o3d.io.read_triangle_mesh(model_path)
    if not mesh.has_vertex_normals():
        print("Mesh has no normals.Computing normals......")
        mesh.compute_vertex_normals()
    #Mesh Smoothing: Apply smoothing algorithms to the mesh for better visual quality.
    mesh = mesh.filter_smooth_simple(number_of_iterations=5)
    #o3d.visualization.draw_geometries([mesh], window_name="3D Model Viewer") ## Use this to visualize when not using google colab
    o3d.visualization.draw_plotly([mesh], window_name="3D Model Viewer")
visualize_obj_model("/content/TripoSR/output/0/mesh.obj")

[Open3D WARNING] Unable to load file /content/TripoSR/output/0/mesh.obj with ASSIMP: Unable to open file "/content/TripoSR/output/0/mesh.obj".
Mesh has no normals.Computing normals......


[Optional]Create a .stl file

In [ ]:
obj_file = "/content/TripoSR/output/0/mesh.obj"
# Load the .obj mesh
ms = pymesh.MeshSet()
ms.load_new_mesh(obj_file)
mesh = ms.current_mesh()
# Convert to .stl format
stl_file = 'model.stl'
ms.save_current_mesh(stl_file)